## Demonstration and Testing of Time-Varying Mean capabilities

In [11]:
import numpy as np
from datetime import datetime, timedelta
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.transition.linear import RandomWalk
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.driver import  AlphaStableNSMDriver
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.models.transition.linear import ConstantVelocity

In [12]:
mu_model='RW'
run_inference=True
avoid_subintervals=True
show_1D=True
save_1D=True
plot_tracks=True
if plot_tracks:
    uncertainty=True
    particle=False
    plot_particle_paths=False

In [13]:
start_time = datetime.now().replace(microsecond=0)

seed = 1 # Random seem for reproducibility

#time-varying skew parameters
sigma_mu=0.0005
q=sigma_mu**2
if mu_model=='RW':
    initial_mu_W_x= +0.001
    initial_mu_W_y= -0.001
    RW_mu_driver= RandomWalk(noise_diff_coeff=q) #1D GRW
    mu_driver = RW_mu_driver
elif mu_model=='CV':
    q/=10
    initial_mu_W_x= np.array([[0.00],[0]])
    initial_mu_W_y= np.array([[0.00],[0]])
    CV_mu_driver=ConstantVelocity(noise_diff_coeff=q)
    mu_driver =CV_mu_driver

# Driving process parameters and drivers
sigma_W2 = 0.00025
alpha = 1.9
c=10
driver_x = AlphaStableNSMDriver(mu_W=initial_mu_W_x, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, mu_W_transition_model=mu_driver ,mu_W_state=True)
driver_y = AlphaStableNSMDriver(mu_W=initial_mu_W_y, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, mu_W_transition_model=mu_driver ,mu_W_state=True)

# transition params and model
theta=0.05
num_steps = 200
number_particles = 2000
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])

#measurement params and model
k_v=100
measurement_model = LinearGaussian(
    ndim_state=4,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 2),  # Mapping measurement vector index to state index
    noise_covar=np.array([[sigma_W2*k_v**2, 0],  # Covariance matrix for Gaussian PDF
                          [0, sigma_W2*k_v**2]])
    )

In [14]:
timesteps = [start_time]
x_mu_data=[]
y_mu_data=[]

truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])

for k in range(num_steps):
    timesteps.append(start_time+timedelta(seconds=1*(k+1)))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k+1]))
    x_mu_data.append(transition_model.mu_W[0,0,0])
    y_mu_data.append(transition_model.mu_W[0,0,1])
    # need to explain in tutorial that shape is n x m x num_drivers, 
    # where for us each driver is m=1 but we have 2 of them in the combined model

In [15]:
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
folder_path=rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\TimeVaryingPlots"

In [ ]:
from stonesoup.types.array import StateVectors
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type


# Sample from the prior Gaussian distribution, input MxN 
if mu_model=='RW':
    mu_prior_x = np.atleast_2d(multivariate_normal.rvs(initial_mu_W_x,
                                np.diag([q]),
                                size=number_particles))
    mu_prior_y = np.atleast_2d(multivariate_normal.rvs(initial_mu_W_y,
                                np.diag([q]),
                                size=number_particles))
    mu_driver = RW_mu_driver #1D GRW
elif mu_model=='CV':
    mu_prior_x = multivariate_normal.rvs(initial_mu_W_x.flatten(),
                                np.diag([q,0]),
                                size=number_particles).T
    mu_prior_y = multivariate_normal.rvs(initial_mu_W_y.flatten(),
                                np.diag([q,0]),
                                size=number_particles).T
    mu_driver= CV_mu_driver

if avoid_subintervals:
    mu_W_state=None
else:
    mu_W_state=True

driver_x = AlphaStableNSMDriver(mu_W=mu_prior_x, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, mu_W_transition_model=mu_driver,mu_W_state=mu_W_state)
driver_y = AlphaStableNSMDriver(mu_W=mu_prior_y, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, mu_W_transition_model=mu_driver,mu_W_state=mu_W_state)
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])

In [17]:
if run_inference:
    from stonesoup.types.detection import Detection
    measurements = []
    for state in truth:
        measurement = measurement_model.function(state, noise=True)
        measurements.append(Detection(measurement,
                                    timestamp=state.timestamp,
                                    measurement_model=measurement_model))
        
    from stonesoup.predictor.particle import MarginalisedParticlePredictor
    from stonesoup.resampler.particle import SystematicResampler 
    from stonesoup.updater.particle import MarginalisedParticleUpdater

    predictor = MarginalisedParticlePredictor(transition_model=transition_model)
    resampler = SystematicResampler()
    updater = MarginalisedParticleUpdater(measurement_model, resampler)

    from stonesoup.types.state import MarginalisedParticleState
    from stonesoup.types.hypothesis import SingleHypothesis
    from stonesoup.types.track import Track

    # Sample from the prior Gaussian distribution
    states = multivariate_normal.rvs(np.array([0, 1, 0, 1]),
                                    np.diag([1., 1., 1., 1.]),
                                    size=number_particles)
    covars = np.stack([np.eye(4) * 100 for i in range(number_particles)], axis=2) # (M, M, N)
    # Create prior particle state.
    prior = MarginalisedParticleState(
        state_vector=StateVectors(states.T),
        covariance=covars,
        weight=np.array([Probability(1/number_particles)]*number_particles),
                        timestamp=start_time-timedelta(seconds=1))

    track = Track()
    x_mu_estimate = []
    y_mu_estimate = []
    for measurement in measurements:
        prediction = predictor.predict(prior, timestamp=measurement.timestamp)
        hypothesis = SingleHypothesis(prediction, measurement)
        post = updater.update(hypothesis)
        track.append(post)
        prior = track[-1]   
        x_mu_estimate.append(np.mean(transition_model.mu_W[0,0,0]))
        y_mu_estimate.append(np.mean(transition_model.mu_W[0,0,1]))
        print(f"track length ={len(track)} of {len(measurements)}")

    from stonesoup.smoother.particle import MarginalisedKalmanSmoother, ParticleSmoother, CarterKohnSmoother
    particlesmoother=ParticleSmoother()
    culled_track=particlesmoother.particle_paths(track=track)
    RTSsmoother=MarginalisedKalmanSmoother()
    RTS_track=RTSsmoother.smooth(track=track)
    CKsmoother=CarterKohnSmoother()
    CK_track=CKsmoother.smooth(track= track)

track length =1 of 201
track length =2 of 201
track length =3 of 201
track length =4 of 201
track length =5 of 201
track length =6 of 201
track length =7 of 201
track length =8 of 201
track length =9 of 201
track length =10 of 201
track length =11 of 201
track length =12 of 201
track length =13 of 201
track length =14 of 201
track length =15 of 201
track length =16 of 201
track length =17 of 201
track length =18 of 201
track length =19 of 201
track length =20 of 201
track length =21 of 201
track length =22 of 201
track length =23 of 201
track length =24 of 201
track length =25 of 201
track length =26 of 201
track length =27 of 201
track length =28 of 201
track length =29 of 201
track length =30 of 201
track length =31 of 201
track length =32 of 201
track length =33 of 201
track length =34 of 201
track length =35 of 201
track length =36 of 201
track length =37 of 201
track length =38 of 201
track length =39 of 201
track length =40 of 201
track length =41 of 201
track length =42 of 201
t

In [18]:
trajectory_scales=[5e+4,1e+2,1e+5,1e+2]

In [19]:
from plotly.subplots import make_subplots
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
axis_label_list=["X1(t)","dX1(t)_dt","X2(t)","dX2(t)_dt"]

particle_plotter_dict = {}
for i,label in enumerate(axis_label_list):
    if plot_tracks:
        file_path = Path(folder_path + rf"\1D_plot_{label}_incl_tracks.html")
    else:
        file_path = Path(folder_path + rf"\1D_plot_{label}_.html")
    file_path.parent.mkdir(parents=True, exist_ok=True)

    particle_plotter_dict[label]= Plotterly(autosize=False, width=1500,height=800,dimension=Dimension.ONE, axis_labels=[label])
    colorway=particle_plotter_dict[label].fig.layout.colorway

    particle_plotter_dict[label].fig = make_subplots(specs=[[{"secondary_y": True}]])
    particle_plotter_dict[label].fig.layout.colorway = colorway
    particle_plotter_dict[label].plot_ground_truths(truth, [i],mode="lines", line=dict(width=2,color='red',dash='dash'))
    
    if i<=1:
        data=x_mu_data
        data_estimate=x_mu_estimate
    else:
        data=y_mu_data
        data_estimate=y_mu_estimate
    particle_plotter_dict[label].fig.add_scatter(y=np.array(data), x=timesteps, name='Groundtruth mu_W', secondary_y=True,line=dict(color='green',width=2))
    particle_plotter_dict[label].fig.add_scatter(y=np.zeros_like(data), x=timesteps, name=f'mu_W, = 0',secondary_y=True,line=dict(color="green",dash='dash'))
    particle_plotter_dict[label].fig.add_scatter(y=np.zeros_like(data), x=timesteps, name=f'{label} = 0',secondary_y=False,line=dict(color="orange",dash='dash'))
    particle_plotter_dict[label].fig.add_scatter(y=np.array(data_estimate), x=timesteps, name='mu_W estimate', secondary_y=True,line=dict(color='pink',width=2))

    if plot_tracks:
        if i==0 or i==2:
            particle_plotter_dict[label].plot_measurements(measurements, [i],marker=dict(symbol="x",size=4))
        particle_plotter_dict[label].plot_tracks(track, [i], mode="lines", uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Filtered")
        particle_plotter_dict[label].plot_tracks(culled_track, [i],mode="lines", uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="\'Descendant\'")
        particle_plotter_dict[label].plot_tracks(RTS_track, [i],mode="lines", uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths,track_label="RTS")
        particle_plotter_dict[label].plot_tracks(CK_track,[i],mode="lines", uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths,track_label="CK")

    
    max_mu=np.max(np.abs(data))*1.1
    max_trajectory=np.max(np.abs(truth[:].state_vector[i]))
    # max_trajectory=max_mu*trajectory_scales[i]
    particle_plotter_dict[label].fig.update_yaxes(
            secondary_y=False,
            title_text=f"{label}",
            mirror=True,
            ticks='outside',
            showline=True,
            linecolor='black',
            gridcolor='lightgrey',
            range=[-max_trajectory,max_trajectory],
            title=dict(text=label, font=dict(size=20))
    )
    particle_plotter_dict[label].fig.update_yaxes(
            secondary_y=True,
            title_text="mu_W",
            mirror=True,
            ticks='outside',
            showline=True,
            linecolor='black',
            gridcolor='lightgrey',
            range=[-max_mu,max_mu],
            title=dict(text="Time", font=dict(size=20))
    )
    particle_plotter_dict[label].fig.update_layout(
                plot_bgcolor='white',
        legend=dict(
                    font=dict(size=15),       # Make the legend font larger
                    # orientation='v',
                    # xanchor="auto",         # Center the legend
                    # yanchor="auto",           # Align the legend to the bottom of the plot
                    bordercolor="Black",
                    borderwidth=3,
                    # y=+0.45,                   # Position it above the graph
                    # x=0.6                    # Center it horizontally
                ),)
    particle_plotter_dict[label].fig.update_xaxes(
                title=dict(text="Time", font=dict(size=20)),
                mirror=True,
                ticks='outside',
                showline=True,
                linecolor='black',
                gridcolor='lightgrey',
            )

    
    if save_1D:
        particle_plotter_dict[label].fig.write_html(str(file_path))
    if show_1D:
        particle_plotter_dict[label].fig.show()